In [12]:
import numpy as np

In [13]:
prefix = "assets/ncc/"
best_matching_corner_list = np.load(prefix + "best_matching_corner_list.npy", allow_pickle=True)
second_best_matching_corner_list = np.load(prefix + "second_best_matching_corner_list.npy", allow_pickle=True)

In [14]:
print("Best matching corner list")
print(best_matching_corner_list[:5])

print("\nSecond best matching corner list")
print(second_best_matching_corner_list[:5])

Best matching corner list
[[(np.int64(11), np.int64(91)) (np.int64(11), np.int64(21))
  np.float64(1.0)]
 [(np.int64(14), np.int64(92)) (np.int64(14), np.int64(22))
  np.float64(1.0000000000000002)]
 [(np.int64(19), np.int64(86)) (np.int64(19), np.int64(16))
  np.float64(1.0000000000000002)]
 [(np.int64(19), np.int64(131)) (np.int64(19), np.int64(61))
  np.float64(0.9999999999999999)]
 [(np.int64(20), np.int64(140)) (np.int64(20), np.int64(70))
  np.float64(1.0000000000000002)]]

Second best matching corner list
[[(np.int64(14), np.int64(92)) (np.int64(11), np.int64(21))
  np.float64(0.14847027130153428)]
 [(np.int64(19), np.int64(86)) (np.int64(11), np.int64(152))
  np.float64(0.278556496704178)]
 [(np.int64(19), np.int64(131)) (np.int64(11), np.int64(21))
  np.float64(0.022869000639856467)]
 [(np.int64(20), np.int64(140)) (np.int64(11), np.int64(21))
  np.float64(0.3920550633189286)]
 [(np.int64(25), np.int64(82)) (np.int64(19), np.int64(16))
  np.float64(0.5820105431604901)]]


In [ ]:
# def filter_corner_pairs_using_NCC_ratio(best_matching_corner_list, second_best_matching_corner_list):
#     matched_corner_pairs = []
#     for i in range(len(best_matching_corner_list)):
#         bl, br, bNCC = best_matching_corner_list[i]
#         # print(f"Best matching corner: ({bl}, {br}), NCC: {bNCC}\n")
        
#         has_second_best_match = False
#         for j in range(len(second_best_matching_corner_list)):
#             sl, sr, sNCC = second_best_matching_corner_list[j]
#             # print(f"Second best matching corner: ({sl}, {sr}), NCC: {sNCC}\n")
            
#             if sl == bl:
#                 has_second_best_match = True
#                 r = sNCC / bNCC
#                 # print(f"NCC ratio: {r}\n")
#                 if r < 0.9 and r >0:
#                     matched_corner_pairs.append((bl, br))
                    
#         if not has_second_best_match:
#             matched_corner_pairs.append((bl, br))
#     return matched_corner_pairs


# def filter_corner_pairs_using_NCC_ratio(best_matching_corner_list, second_best_matching_corner_list):
#     matched_corner_pairs = []
    
#     # Convert the second-best list into a dictionary for easier lookup
#     second_best_dict = {sl: (sr, sNCC) for sl, sr, sNCC in second_best_matching_corner_list}
    
#     for bl, br, bNCC in best_matching_corner_list:
        
#         # Check if this left corner has a recorded second-best match
#         if bl in second_best_dict:
#             _sr, sNCC = second_best_dict[bl]
            
#             # Compute the ratio
#             r = sNCC / bNCC
            
#             # Apply the exclusive bounds check (0 < r < 0.9)
#             if 0 < r < 0.9:
#                 matched_corner_pairs.append((bl, br))
                
#     return matched_corner_pairs

# This function assumes input lists have same structure (length, and (lx, ly) order)
# but the given input data is not!!!
def filter_corner_pairs_using_NCC_ratio(best_matching_corner_list, second_best_matching_corner_list):
    matched_corner_pairs = []
    for i in range(len(second_best_matching_corner_list)):
        _sl, _sr, sNCC = second_best_matching_corner_list[i]
        bl, br, bNCC = best_matching_corner_list[i]
        r = sNCC / bNCC
        # print(f"NCC ratio: {r}\n")
        if r < 0.9 and r >0:
            matched_corner_pairs.append((bl, br))
    return matched_corner_pairs

In [16]:
matched_corner_pairs = filter_corner_pairs_using_NCC_ratio(best_matching_corner_list, second_best_matching_corner_list)


In [17]:
expected_matched_corner_pairs = np.load(prefix + "filtered_matched_corner_pairs.npy", allow_pickle=True)

In [18]:
print(len(expected_matched_corner_pairs))
expected_matched_corner_pairs[:5]

96


array([[[ 11,  91],
        [ 11,  21]],

       [[ 14,  92],
        [ 14,  22]],

       [[ 19,  86],
        [ 19,  16]],

       [[ 19, 131],
        [ 19,  61]],

       [[ 20, 140],
        [ 20,  70]]])

In [19]:
print(len(matched_corner_pairs))
np.array(matched_corner_pairs[:5])

96


array([[[ 11,  91],
        [ 11,  21]],

       [[ 14,  92],
        [ 14,  22]],

       [[ 19,  86],
        [ 19,  16]],

       [[ 19, 131],
        [ 19,  61]],

       [[ 20, 140],
        [ 20,  70]]])

In [23]:
np.array_equal(matched_corner_pairs, expected_matched_corner_pairs)

True

In [20]:
def matches_to_rows(matches):
    return np.array(
        [[lx, ly, rx, ry] for [lx, ly], [rx, ry] in matches],
        dtype=float,
    )

def save_matches_to_csv(matches, filename):
    rows = matches_to_rows(matches)
    np.savetxt(
        filename,
        rows,
        delimiter=",",
        fmt=["%d", "%d", "%d", "%d"],
    )
save_matches_to_csv(matched_corner_pairs, "tmp/matched_corner_pairs.csv")
save_matches_to_csv(expected_matched_corner_pairs, "tmp/expected_matched_corner_pairs.csv")

In [21]:
# from collections import Counter

# # Normalize nested tuples and numpy scalar types so set/counter operations are reliable.
# def _to_py_tuple(x):
#     if isinstance(x, np.ndarray):
#         x = x.tolist()
#     if isinstance(x, (list, tuple)):
#         return tuple(_to_py_tuple(v) for v in x)
#     if isinstance(x, np.generic):
#         return x.item()
#     return x

# exp = [_to_py_tuple(v) for v in expected_matched_corner_pairs]
# got = [_to_py_tuple(v) for v in matched_corner_pairs]

# exp_set = set(exp)
# got_set = set(got)

# only_in_expected = sorted(exp_set - got_set)
# only_in_matched = sorted(got_set - exp_set)

# print("Only in expected:", len(only_in_expected))
# print("Only in matched:", len(only_in_matched))

# print("\nSample only in expected:", only_in_expected[:10])
# print("Sample only in matched:", only_in_matched[:10])

# # Optional: check multiplicity differences (if duplicates exist)
# exp_counter = Counter(exp)
# got_counter = Counter(got)
# count_mismatch = {
#     k: (exp_counter[k], got_counter[k])
#     for k in (exp_set | got_set)
#     if exp_counter[k] != got_counter[k]
# }
# print("\nElements with different counts:", len(count_mismatch))


## Check error on given data

In [22]:
# Full review at 
# https://docs.google.com/spreadsheets/d/1RMdmltZzZc8qnQLAJYsmNA8eoa436dTh-2BSrL4j3gU/edit?gid=1253851682#gid=1253851682

print("10th element of best_matching_corner_list")
print(best_matching_corner_list[9])

print("\n9th element of second_best_matching_corner_list")
print(second_best_matching_corner_list[8]) # Should this be the [9]th element?

r = second_best_matching_corner_list[8][2] / best_matching_corner_list[9][2]
print(f"\nNCC ratio: {r}")

print("\n10th element of expected_matched_corner_pairs")
print(expected_matched_corner_pairs[9])

10th element of best_matching_corner_list
[(np.int64(54), np.int64(37)) (np.int64(88), np.int64(159))
 np.float64(0.5652114311008104)]

9th element of second_best_matching_corner_list
[(np.int64(54), np.int64(37)) (np.int64(52), np.int64(170))
 np.float64(0.5317844915709701)]

NCC ratio: 0.9408594064264806

10th element of expected_matched_corner_pairs
[[ 54  37]
 [ 88 159]]
